# L10a: Online Learning: Weighted Majority and Multiplicative Weights Algorithms
In this lecture, we introduce online learning and several algorithms that build upon following the advice of experts and providing feedback to the experts based on their performance.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe the online learning problem:__ [Online learning](https://en.wikipedia.org/wiki/Online_machine_learning) is a type of machine learning where the model learns from data as it arrives, rather than from a fixed dataset. We'll set up online learning as a game played over many rounds, in which we make a prediction, an adversary reveals the outcome, and we learn from the result.
> * __Measure an online learning algorithm with regret:__ [Regret](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-6/L6a/docs/Orabona-OnlineLearning-AnnRev-2021.pdf) is the difference between the loss of an algorithm and the loss of the best fixed strategy in hindsight. We'll define the regret of an online learning algorithm and see why we want a bound on regret that grows more slowly than the number of rounds.
> * __Learn from expert advice with multiplicative weights:__ The [weighted majority and multiplicative weights algorithms](https://en.wikipedia.org/wiki/Multiplicative_weight_update_method) keep a weight for each expert and shrink the weight of an expert each time it does poorly. We'll work through both algorithms and the bounds that compare their performance with that of the best expert.

This is our first lecture on online learning, which is a foundational component of reinforcement learning. Online learning is such an intuitively simple idea: a machine (like us from when we are a baby to adulthood) learns from experience, good and bad. The idea dates [back to the 1950s and has been studied extensively in the context of game theory, optimization, and machine learning](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-6/L6a/docs/Orabona-OnlineLearning-arXiv-2019.pdf).

I'm super excited to share this module with you, so let's get started!
___

## General Reinforcement/Online Learning Problem
Suppose we have an agent that can be in a state $s \in \mathcal{S}$, where $\mathcal{S}$ is the set of possible states, and can take an action $a \in \mathcal{A}$, where $\mathcal{A}$ is the set of possible actions. After taking action $a$ in state $s$, the agent receives a reward $r$. But how does the agent learn to choose the best possible action in each state to maximize its cumulative reward over time?

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Schematic-RL.svg" width="580" style="max-width:100%; height:auto;" alt="Loop diagram. A blue Agent box at the top sends an arrow labeled action a down the right side to a green Environment box at the bottom. Two arrows return up the left side from the Environment to the Agent, labeled reward r and state s to s prime."/>
</p>

In reinforcement learning, an agent interacts with an environment by observing its current state $s \in \mathcal{S}$, selecting an action $a \in \mathcal{A}$, and receiving a reward that influences its future decisions. We'll explore three distinct approaches to this problem:

* __Multiplicative weights__ update expert weights based on their past performance.
* __Bandit algorithms__ operate in stateless environments. On each round, they explore different actions to estimate their rewards and adapt their action-selection strategy based on the outcomes.
* __Q-learning__ is a value-based method that estimates the long-term value of each state-action pair, enabling the agent to learn optimal behavior in environments with temporal and sequential dynamics.


The feedback available to the agent determines how it can learn. With expert advice, the revealed outcome lets us score every expert after each round. With bandit feedback, we observe only the reward of the action we chose, so trying an action also supplies information about its reward distribution. In Q-learning, actions also change the state from which future decisions are made.

___

## Online Learning and Regret Minimization
Suppose we are playing a number-guessing game where we have to predict a number over $T$ rounds. 

For $t = 1, 2, ..., T$ __do__:
1. An _adversary_ picks a number $y_t$ from the set $\{1, 2, ..., N\}$ and keeps it secret.
2. The _aggregator_ (us) makes a prediction $\hat{y}_t$ for the number, based on some strategy or algorithm.
3. The adversary reveals the true number $y_t$, and we receive feedback using a loss function $l_t(\hat{y}_t, y_{t})$, which measures the discrepancy between our prediction and the true number.

The cumulative difference between what we selected at each turn and the __best fixed strategy in hindsight__ is called the _regret_:
$$
\begin{equation*}
R(T) = \underbrace{\sum_{t=1}^{T} l_t(\hat{y}_t, y_t)}_{\text{aggregator}} - \underbrace{\min_{u} \sum_{t=1}^{T} l_t(u,y_t)}_{\text{best fixed strategy in hindsight}},
\end{equation*}
$$
where the minimum is taken over every fixed prediction $u$ that we could have made in all $T$ rounds.

The objective of online learning is to design algorithms that _minimize cumulative regret_, ensuring that our online learning algorithm performs _nearly_ as well as the best fixed strategy in hindsight.

> __Key distinctions__
> 
> * __Adaptation__: Online learning algorithms are designed to adapt to changing data distributions and learn from their mistakes, while traditional supervised machine learning algorithms are trained on a fixed dataset.
> * __Regret minimization__: Regret is a property of the _algorithm_ that the player uses. The goal of an algorithm designer is to produce an algorithm that minimizes regret.

Now that we have a basic understanding of online learning and regret minimization, let's explore a simple implementation of this idea, the weighted majority algorithm.

___

## Weighted Majority Algorithm (WMA)
Let's start with the __Weighted Majority Algorithm (WMA)__ developed by Littlestone and Warmuth in 1994:
* [Littlestone, N., & Warmuth, M. K. (1994). The weighted majority algorithm. Information and Computation, 108(2), 212-261.](https://doi.org/10.1006/inco.1994.1009)

We illustrate this approach with an example found in [Arora et al., 2005, Princeton](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-6/L6a/docs/Arora-MWsurvey-CS-Princeton.pdf), which is a _prediction from expert advice_ problem. Let's take a look at the framing: 

* __Game__: Suppose we want to predict the daily movement of a stock price, i.e., whether it will go `{up | down}` on the day; in this case, we are predicting a _binary_ outcome which we encode as  `{1 | -1}`. We predict whether the price will go up or down each morning. At the market close, we find out the `true` movement of the stock price for the day. _If our prediction is incorrect, we lose a dollar_. Thus, our objective is to _minimize_ our losses throughout the game. However, we don't do this on our own. In making our predictions, we watch the predictions of $n$ experts (whose predictions may be correlated and who may or may not be correct). 
* __Goal__: The weighted majority algorithm (us) must bound our mistakes by a multiple of the _best_ expert's mistakes plus an initial term. However, we do this without knowing which expert is the best, i.e., it is not known until the end of the sequence of days who the best expert is, but we must make decisions each day. The algorithm does this by maintaining a weighting of the experts. Initially, all have equal weight. As time passes, some experts are seen as making better predictions than others, and their share of the total weight grows.

We play this game between an omniscient _adversary_ (nature, i.e., the market) and an _aggregator_ (us) who $n$ experts advise; if the weighted vote is tied, we predict `-1`. The game proceeds in $T$ rounds $t = 1, 2, \ldots, T$. During each round, the aggregator (us) makes a _binary_ prediction $\hat{y}_t \in \{-1, 1\}$, and the adversary (market) reveals the true outcome $y_t$. Initially, the experts have weights $\left\{w_{i}^{(1)} = 1 \mid i = 1, 2, \ldots, n\right\}$. 

### Algorithm
For each round $t=1,2,\dots,T$ __do__:
1. The aggregator (us) makes a prediction $\hat{y}_t \in \{-1, 1\}$ based on the weighted majority of the experts' predictions. If the total weight of the experts predicting `1` at time $t$ is greater than the total weight of the experts predicting `-1`, then the aggregator predicts `1`; otherwise, it predicts `-1`.
2. The adversary (market) reveals the actual outcome $y_t \in \{-1, 1\}$.
3. We decrease the weights of the experts who predicted incorrectly. For each expert $i$ who predicted incorrectly, we update the weight: $w_{i}^{(t+1)} = w_{i}^{(t)}(1-\epsilon)$, where $0<\epsilon\leq{1/2}$ is a learning rate parameter.

### Worked Example: Four Rounds with Three Experts
Suppose three experts predict a binary outcome over four rounds. We use $\epsilon=1/2$, so each mistake halves an expert's weight. The table gives the weights **before** the round, the expert predictions, our weighted vote, and the revealed outcome:

| Round | Weights before the round | Expert predictions | Our prediction | Actual outcome |
|---|---|---|---:|---:|
| 1 | (1, 1, 1) | (1, 1, −1) | 1 | 1 |
| 2 | (1, 1, 0.5) | (−1, 1, 1) | 1 | −1 |
| 3 | (1, 0.5, 0.25) | (−1, 1, 1) | −1 | 1 |
| 4 | (0.5, 0.5, 0.25) | (1, −1, 1) | 1 | 1 |

In round 2, the experts predicting up carry weight $1+0.5=1.5$, while the expert predicting down carries weight 1. We predict up and make a mistake. Experts 2 and 3 also make a mistake, so their weights become 0.5 and 0.25. In round 3, those two experts again predict up, but their combined weight is now only 0.75. Expert 1 carries weight 1, so the weighted vote predicts down.

After round 4, the weights are $(0.5,0.25,0.25)$. Expert 1 made one mistake; experts 2 and 3 each made two. We made two mistakes, in rounds 2 and 3. With a loss of one per mistake, our regret against the best fixed expert is given by:
$$
R_{\mathrm{WMA}}(4)=2-\min\{1,2,2\}=1.
$$
The weights identify the best expert at the end, while our earlier votes still contribute to our cumulative loss. The theorem bounds that loss for any sequence of expert predictions and outcomes.

### Mistake Guarantee

__Theorem__: The weighted majority algorithm has the following theoretical guarantee (which bounds the number of mistakes the aggregator makes). Let $M_{i}^{(t)}$ be the number of mistakes made by expert $i$ up to time $t$ and $M^{(t)}$ be the total number of mistakes made by the aggregator (us). Then, for every expert $i$ and the aggregator, we have:
$$
\begin{align*}
M^{(t)} \leq \frac{2\ln(n)}{\epsilon} + 2\left(1+\epsilon\right)M_{i}^{(t)}.
\end{align*}
$$

The [proof of this theorem can be found here](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-6/L6a/docs/Arora-MWsurvey-CS-Princeton.pdf). The weighted majority algorithm is deterministic and has a simple implementation (any randomness comes from the adversary). Its guarantee is __multiplicative__: our mistakes are at most about twice the best expert's mistakes, plus an initial term. Next, we choose an expert at random, which gives an __additive__ guarantee: our expected cost exceeds the best expert's cost by at most a term that grows more slowly than the number of rounds.

___

## Multiplicative Weights Algorithm (MWA)
The **Multiplicative Weights Algorithm (MWA)** is a simple yet robust online learning method that embodies a similar idea to the weighted majority algorithm, i.e., learning from expert advice. Here, the learning rate $\eta$ plays a role analogous to $\epsilon$ in the Weighted Majority Algorithm, controlling adaptation speed. 

Let’s walk through the setup and sketch out the algorithm.

### Problem Setting
Suppose we are faced with a repeated decision-making task over rounds $t = 1, 2, \ldots, T$. At each round, we have access to $n$ experts, each providing a recommendation or prediction. Our goal is to combine their advice adaptively in order to make strong decisions over time, even in adversarial or uncertain environments. We assume the adversary fixes the advice and the outcomes in advance, so they do not depend on which expert we draw.

* Let $\mathbf{p}^{(t)} = \{p_1^{(t)}, p_2^{(t)}, \ldots, p_n^{(t)}\}$ denote our belief distribution over experts at round $t$, updated iteratively based on their past performance.
* We select an expert by sampling from this distribution, for example, using a Categorical distribution:
  $i \sim \texttt{Categorical}(\mathbf{p}^{(t)})$, and follow that expert’s recommendation.
* After the decision is made, the environment (or adversary) reveals the true outcome. We then compute a cost vector $\mathbf{m}^{(t)} = \{m_1^{(t)}, \dots, m_n^{(t)}\}$, where $m_i^{(t)} \in [-1, 1]$ denotes the cost incurred by expert $i$ at time $t$. A correct prediction receives a cost of $-1$, and an incorrect one receives a cost of $+1$.

The **expected cost** of the algorithm at round $t$ is given by:
$$
L^{(t)} = \sum_{i=1}^{n} p_i^{(t)} m_i^{(t)}.
$$
The **total expected cost** over $T$ rounds is given by:
$$
L_A = \sum_{t=1}^{T} L^{(t)} = \sum_{t=1}^{T} \sum_{i=1}^{n} p_i^{(t)} m_i^{(t)}.
$$
The goal of the algorithm is to ensure that this cumulative loss is as close as possible to that of the **best expert in hindsight**, this difference we give a 
special name, the **regret**:
$$
R(T) = \underbrace{\sum_{t=1}^{T} \sum_{i=1}^{n} p_i^{(t)} m_i^{(t)}}_{\text{Algorithm's Loss}} - \underbrace{\min_{i \in \{1, \dots, n\}} \sum_{t=1}^{T} m_i^{(t)}}_{\text{Best Expert's Loss}}.
$$
This quantity measures how much worse the algorithm performs compared to the best fixed decision rule. A key property of successful online learning algorithms like MWA is a **sublinear** regret bound, i.e., a bound that grows more slowly than the number of rounds $T$. Then the average regret per round, $R(T)/T$, is at most a quantity that vanishes as $T\to\infty$.


### Algorithm

__Initialize__: Fix a learning rate $0<\eta\leq{1}/{2}$, for each expert initialize the weight $w_{i}^{(1)} = 1$.

For $t=1,2,\dots,T$ __do__:
1. Ask each expert $i$ what the outcome of the experiment should be, and denote the expert's answer as $\hat{y}_{i}^{(t)}$. Choose expert $i$ with probability $p_{i}^{(t)} = w_{i}^{(t)}/\sum_{j=1}^{n}w_{j}^{(t)}$ and follow its answer.
2. The adversary (nature) reveals the true outcome $y_{t}$ of the experiment at time $t$. Compute the cost $m_{i}^{(t)}$ of each expert $i$, using that expert's answer $\hat{y}_{i}^{(t)}$:
    $$
    m_i^{(t)} =
    \begin{cases}
    -1 & \text{if } \hat{y}_i^{(t)} = y_t \quad \text{(correct)} \\
    +1 & \text{if } \hat{y}_i^{(t)} \neq y_t \quad \text{(incorrect)}
    \end{cases}.
   $$
3. Update the weight of each expert $i$ (renormalize the weights to obtain the new probability distribution):
$$
\begin{align*}
w_{i}^{(t+1)} = w_{i}^{(t)}\cdot\left(1-\eta\cdot{m_{i}^{(t)}}\right).
\end{align*}
$$

This is a super simple algorithm, with some very nice properties. The weights are updated multiplicatively based on the performance of each expert, hence the name Multiplicative Weights Algorithm. The learning rate $\eta$ controls how aggressively the algorithm adapts to the experts' performance. And there is a theoretical guarantee that the algorithm will perform nearly as well as the best fixed expert in hindsight, let's check that out!

### Theoretical Regret Bound
Assume all costs lie in the range $m_i^{(t)} \in [-1, 1]$, and fix a learning rate $0<\eta \leq \frac{1}{2}$. Then the Multiplicative Weights Algorithm (MWA) guarantees that for any expert $i$, after $T$ rounds, the first line below holds. Taking $i$ to be the best expert in hindsight turns the left side of the second line into the regret:
$$
\begin{align*}
\sum_{t=1}^{T} \mathbf{p}^{(t)} \cdot \mathbf{m}^{(t)} & \leq \sum_{t=1}^{T} m_i^{(t)} + \eta \sum_{t=1}^{T} |m_i^{(t)}| + \frac{\ln n}{\eta} \\
\underbrace{\sum_{t=1}^{T} \mathbf{p}^{(t)} \cdot \mathbf{m}^{(t)} - \overbrace{\sum_{t=1}^{T} m_i^{(t)}}^{\text{best expert}}}_{R(T)} & \leq \eta T + \frac{\ln n}{\eta} \\
R(T) & \leq \eta T + \frac{\ln n}{\eta},
\end{align*}
$$
where we used $|m_i^{(t)}|\leq1$; equality holds for our binary signed costs. By choosing $\eta = \sqrt{\frac{\ln n}{T}}$, which satisfies $\eta\leq{1/2}$ when $T\geq 4\ln n$, this regret bound becomes sublinear:
$$
R(T) \leq 2 \sqrt{T \ln n}.
$$
Thus, the average regret per round is at most $2\sqrt{\ln n/T}$, which vanishes as $T\to\infty$, so MWA performs nearly as well as the best fixed expert in hindsight.

### Compare the Two Guarantees
Both algorithms use **full feedback**: once the outcome is revealed, we score and update every expert. They differ in how they choose a prediction and in the performance quantity they bound:

| Algorithm | Prediction rule | Performance guarantee |
|---|---|---|
| Weighted majority | Take the weighted binary vote | Our mistakes are bounded by roughly twice the best expert's mistakes, plus an initial term. |
| Multiplicative weights | Sample an expert in proportion to its weight | Our expected cumulative cost exceeds the best expert's cost by at most a term proportional to the square root of the number of rounds, with the stated learning rate. |

__How do the two loss conventions compare?__ The weighted majority algorithm counts mistakes, while our multiplicative weights example uses signed costs. Let $\ell_i^{(t)}$ equal one if expert $i$ makes a mistake in round $t$ and zero otherwise. Then the signed cost is given by:
$$
m_i^{(t)}=2\ell_i^{(t)}-1.
$$
Over $T$ rounds, each signed-cost total, ours and each expert's, is twice the matching mistake total minus $T$, and the $T$ cancels in the regret. So the signed-cost regret is twice the expected regret counted in mistakes, and the bound becomes $\sqrt{T\ln n}$ when we count mistakes. [Arora et al., Sections 1–2](https://theoryofcomputing.org/articles/v008a006/v008a006.pdf) develop both algorithms in more detail.


### Additional Resources
This module borrowed notes and was inspired from several sources: [Arora et al., The Multiplicative Weights Update Method: A Meta-Algorithm and Applications, Theory of Computing, Volume 8 (2012), pp. 121–164](https://theoryofcomputing.org/articles/v008a006/v008a006.pdf) and the [15-859 CMU Lecture 16](https://www.cs.cmu.edu/afs/cs.cmu.edu/academic/class/15859-f11/www/notes/lecture16.pdf) and [15-850 CMU Lecture 17](https://www.cs.cmu.edu/afs/cs.cmu.edu/academic/class/15859-f11/www/notes/lecture17.pdf).

___

## Lab
In lab `L10b`, we implement the weighted majority algorithm and use it to combine the advice of five experts, each a ticker such as QQQ or AAPL, on whether the SPY exchange-traded fund goes up or down each trading day. We then check the theorem above, first when the experts see the same day's price moves and then when they must forecast the next day.

___

## Summary
In this lecture, we introduced online learning and two algorithms that learn from expert advice.

> __Key Takeaways:__
>
> * __Online learning and regret:__ We set up online learning as a repeated game in which an aggregator makes a prediction, an adversary reveals the outcome, and the aggregator learns from the result. We measured the performance of an algorithm with its regret, the gap between its total loss and that of the best fixed strategy in hindsight, and looked for a regret bound that grows more slowly than the number of rounds.
>
> * __The weighted majority algorithm:__ We kept a weight for each expert, predicted with the weighted majority vote, and reduced the weight of every expert that made a mistake. The theorem bounds the number of mistakes we make by a term that grows with the logarithm of the number of experts plus about twice the mistakes of the best expert, even though we do not know in advance which expert is best.
>
> * __The multiplicative weights algorithm:__ We turned the expert weights into a probability distribution, followed an expert chosen at random from it, and updated every expert's weight by its cost. With a learning rate chosen for the number of rounds, the regret bound grows only with the square root of the number of rounds, so the algorithm does nearly as well as the best expert.

In L10c, the agent no longer sees how every option would have done; it sees only the reward of the action it takes. This leads to the multi-armed bandit problem.

___